# Model 2 Refinement — Qwen2.5-3B-Instruct + LoRA
Fresh-runtime version. Key guarantees preserved from the prior session:

- Base model: Qwen2.5-3B-Instruct; V1 = base + `checkpoint-145` (never modified, never overwritten)
- The **466-case validation set is NOT re-run for inference anywhere in this notebook** — it is loaded once from the attached `model2_validation_466_results.jsonl`
- The **499-case test set is only evaluated once for V1 (reused from prior results) and once for V2 (only if V2 is trained)**
- No `SFTTrainer` is used for the refinement step — training is a custom PyTorch loop (`AdamW`, completion-only loss, grad-accum=8), because `SFTTrainer` was confirmed to silently produce zero weight updates in this environment
- Cell 12 (smoke test) and Cell 13 (full refinement) are **separate, manual** cells. Cell 13 must never be run automatically after Cell 12.


In [2]:
# CELL 1 — Install dependencies
!pip install -q -U \
  "trl==0.23.0" \
  "transformers>=4.56.2,<5" \
  "datasets>=3.0,<4" \
  "accelerate>=1.4.0,<2" \
  "peft>=0.13,<0.18" \
  "bitsandbytes>=0.43" \
  "pandas==2.2.3" 2>&1 | grep -v -E "gradio|diffusers|dependency resolver"

!pip install -q --force-reinstall --no-deps "pyarrow>=16,<20" 2>&1 | grep -v -E "gradio|diffusers|dependency resolver"

print("Dependencies installed.")
print("Note: gradio/diffusers dependency-conflict warnings (if any) are filtered out above — "
      "neither package is used in this notebook (no Gradio UI, no Diffusers pipeline).")

Dependencies installed.
Note: gradio/diffusers dependency-conflict warnings (if any) are filtered out above — neither package is used in this notebook (no Gradio UI, no Diffusers pipeline).


In [3]:
# CELL 2 — Configuration
import os, gc, re, json, math, time, random, shutil, inspect
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import transformers
import peft

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import PeftModel, LoraConfig, prepare_model_for_kbit_training

# -----------------------------------------------------------------------
# Paths - confirmed against the uploaded artifacts.
# checkpoint-145 is the V1 LoRA adapter directory (adapter_model.safetensors,
# adapter_config.json, tokenizer files, trainer_state.json, README.md).
# Upload it to Colab and point CHECKPOINT_145_DIR at it. It is READ-ONLY here.
# -----------------------------------------------------------------------
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
CHECKPOINT_145_DIR = "/content/checkpoint-145"          # V1 adapter - NEVER write here
TRAIN_FILE = "/content/model2_train_final.jsonl"        # 2306 cases
VALID_FILE = "/content/model2_validation_final.jsonl"   # 466 cases
TEST_FILE = "/content/model2_test_final.jsonl"          # 499 cases - untouched benchmark
VALID_RESULTS_FILE = "/content/model2_validation_466_results.jsonl"  # already-computed 466 predictions - NEVER rerun

EXPECTED_TRAIN_COUNT = 2306
EXPECTED_VALID_COUNT = 466
EXPECTED_TEST_COUNT = 499

# Confirmed from trainer_state.json / training_args.bin of the provided run:
# max_length=640, packing=True, completion_only_loss=True, lr=2e-4, 1 epoch,
# per_device_train_batch_size=1, grad_accum=16, optim=paged_adamw_8bit,
# lr_scheduler=cosine, seed=42, best_global_step=145, best eval_loss=0.01626.
V1_MAX_LENGTH = 640
SEED = 42

V2_OUTPUT_DIR = "/content/model2_v2_finetuned"          # new dir, never checkpoint-145
EVAL_OUTPUT_DIR = "/content/model2_v1_v2_eval_outputs"
os.makedirs(EVAL_OUTPUT_DIR, exist_ok=True)

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

SYSTEM_PROMPT_EXPECTED = (
    "You are a banking retention analyst. Given a customer retention case, "
    "identify the most supported reason for risk, cite evidence, assign urgency, "
    "and select exactly one eligible retention action."
)

APPROVED_REASONS = [
    "SERVICE_DISSATISFACTION", "COMPETITOR_MIGRATION", "FEE_DISSATISFACTION",
    "LOW_ENGAGEMENT", "PRODUCT_MISMATCH", "DIGITAL_FRICTION", "FINANCIAL_STRESS",
    "LIFE_STAGE_CHANGE", "TEMPORARY_SEASONAL_CHANGE", "UNKNOWN",
]
APPROVED_REASONS_SET = set(APPROVED_REASONS)

APPROVED_ACTIONS = [
    "MONITOR", "SERVICE_RECOVERY", "COMPLAINT_ESCALATION", "FEE_WAIVER_REVIEW",
    "RM_CALLBACK", "PRODUCT_REVIEW", "CARD_REVIEW", "LOAN_REVIEW",
    "RE_ENGAGEMENT", "FINANCIAL_GUIDANCE",
]
APPROVED_ACTIONS_SET = set(APPROVED_ACTIONS)

APPROVED_URGENCY = ["LOW", "MEDIUM", "HIGH"]
APPROVED_URGENCY_SET = set(APPROVED_URGENCY)

FORBIDDEN_KEYS_ANYWHERE = {"churn_flag"}
REQUIRED_TARGET_KEYS = {
    "primary_reason", "secondary_reasons", "evidence", "urgency",
    "recommended_action", "reasoning_summary",
}
REQUIRED_USER_KEYS = {
    "customer_context", "behavior", "service_evidence", "model1", "eligible_actions",
}
FALLBACK_TRIPLE = ("UNKNOWN", "MEDIUM", "MONITOR")

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU available. This notebook requires a CUDA GPU (e.g. Colab T4).")

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

def clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Configuration loaded.")
print(f"MODEL_NAME={MODEL_NAME}")
print(f"CHECKPOINT_145_DIR={CHECKPOINT_145_DIR} (READ-ONLY, never overwritten)")
print(f"V2_OUTPUT_DIR={V2_OUTPUT_DIR}")
print(f"V1_MAX_LENGTH={V1_MAX_LENGTH}  compute_dtype={compute_dtype}")
print(f"torch={torch.__version__} transformers={transformers.__version__} peft={peft.__version__}")


Configuration loaded.
MODEL_NAME=Qwen/Qwen2.5-3B-Instruct
CHECKPOINT_145_DIR=/content/checkpoint-145 (READ-ONLY, never overwritten)
V2_OUTPUT_DIR=/content/model2_v2_finetuned
V1_MAX_LENGTH=640  compute_dtype=torch.bfloat16
torch=2.11.0+cu128 transformers=4.57.6 peft=0.17.1


In [4]:
# CELL 3 - Dataset / schema validation (train=2306, validation=466, test=499)
def reject_nonstandard_constants(x: str):
    raise ValueError(f"Non-standard JSON constant encountered: {x}")

def load_jsonl_strict(path: str) -> List[Dict[str, Any]]:
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                raise ValueError(f"{path}: blank line at {line_num}")
            try:
                obj = json.loads(line, parse_constant=reject_nonstandard_constants)
            except Exception as e:
                raise ValueError(f"{path}: invalid JSON at line {line_num}: {e}") from e
            records.append(obj)
    if not records:
        raise ValueError(f"{path}: file is empty")
    return records

def walk_values(obj):
    if isinstance(obj, dict):
        for k, v in obj.items():
            yield k, v
            yield from walk_values(v)
    elif isinstance(obj, list):
        for v in obj:
            yield None, v
            yield from walk_values(v)

def contains_forbidden_key(obj, forbidden_keys):
    return any(isinstance(k, str) and k in forbidden_keys for k, _ in walk_values(obj))

def ensure_nonempty_string(x, field_name):
    if not isinstance(x, str) or not x.strip():
        raise ValueError(f"Expected non-empty string for {field_name}, got: {type(x)}")

def validate_example(example: Dict[str, Any], path: str, idx: int) -> Dict[str, Any]:
    if "case_id" not in example:
        raise ValueError(f"{path} example {idx}: missing case_id")
    ensure_nonempty_string(example["case_id"], "case_id")

    if "messages" not in example or not isinstance(example["messages"], list) or len(example["messages"]) != 3:
        raise ValueError(f"{path} example {idx}: expected messages list of length 3")

    system_msg, user_msg, assistant_msg = example["messages"]
    if system_msg.get("role") != "system":
        raise ValueError(f"{path} example {idx}: message[0] must be system")
    if user_msg.get("role") != "user":
        raise ValueError(f"{path} example {idx}: message[1] must be user")
    if assistant_msg.get("role") != "assistant":
        raise ValueError(f"{path} example {idx}: message[2] must be assistant")

    if system_msg["content"] != SYSTEM_PROMPT_EXPECTED:
        raise ValueError(f"{path} example {idx}: unexpected system prompt")

    user_content = user_msg.get("content")
    if not isinstance(user_content, dict) or not user_content:
        raise ValueError(f"{path} example {idx}: user content must be a non-empty dict")
    missing_user_keys = REQUIRED_USER_KEYS - set(user_content.keys())
    if missing_user_keys:
        raise ValueError(f"{path} example {idx}: missing user keys {sorted(missing_user_keys)}")

    eligible_actions = user_content["eligible_actions"]
    if not isinstance(eligible_actions, list) or not eligible_actions:
        raise ValueError(f"{path} example {idx}: eligible_actions must be a non-empty list")
    if any(a not in APPROVED_ACTIONS_SET for a in eligible_actions):
        raise ValueError(f"{path} example {idx}: invalid eligible action(s) {eligible_actions}")

    assistant_content = assistant_msg.get("content")
    if not isinstance(assistant_content, dict) or not assistant_content:
        raise ValueError(f"{path} example {idx}: assistant content must be a non-empty dict")
    missing_target_keys = REQUIRED_TARGET_KEYS - set(assistant_content.keys())
    if missing_target_keys:
        raise ValueError(f"{path} example {idx}: missing assistant target keys {sorted(missing_target_keys)}")

    if assistant_content["primary_reason"] not in APPROVED_REASONS_SET:
        raise ValueError(f"{path} example {idx}: invalid primary_reason {assistant_content['primary_reason']}")
    if assistant_content["urgency"] not in APPROVED_URGENCY_SET:
        raise ValueError(f"{path} example {idx}: invalid urgency {assistant_content['urgency']}")
    if assistant_content["recommended_action"] not in APPROVED_ACTIONS_SET:
        raise ValueError(f"{path} example {idx}: invalid recommended_action {assistant_content['recommended_action']}")
    if assistant_content["recommended_action"] not in eligible_actions:
        raise ValueError(f"{path} example {idx}: recommended_action not in eligible_actions")
    if not isinstance(assistant_content["secondary_reasons"], list):
        raise ValueError(f"{path} example {idx}: secondary_reasons must be a list")
    if any(x not in APPROVED_REASONS_SET for x in assistant_content["secondary_reasons"]):
        raise ValueError(f"{path} example {idx}: invalid secondary_reasons")
    if not isinstance(assistant_content["evidence"], list):
        raise ValueError(f"{path} example {idx}: evidence must be a list")
    ensure_nonempty_string(assistant_content["reasoning_summary"], "reasoning_summary")

    if contains_forbidden_key(example, FORBIDDEN_KEYS_ANYWHERE):
        raise ValueError(f"{path} example {idx}: forbidden key churn_flag found")

    return {"case_id": example["case_id"], "user_content": user_content, "assistant_content": assistant_content}

def load_and_validate(path: str) -> List[Dict[str, Any]]:
    raw = load_jsonl_strict(path)
    return [validate_example(r, path, i) for i, r in enumerate(raw, start=1)]

train_records = load_and_validate(TRAIN_FILE)
valid_records = load_and_validate(VALID_FILE)
test_records = load_and_validate(TEST_FILE)

for name, recs, expected in [("train", train_records, EXPECTED_TRAIN_COUNT),
                              ("validation", valid_records, EXPECTED_VALID_COUNT),
                              ("test", test_records, EXPECTED_TEST_COUNT)]:
    if len(recs) != expected:
        raise ValueError(f"{name} set: expected {expected} cases, found {len(recs)}. "
                          f"Stopping rather than silently proceeding with a mismatched split.")

train_ids = {r["case_id"] for r in train_records}
valid_ids = {r["case_id"] for r in valid_records}
test_ids = {r["case_id"] for r in test_records}
assert train_ids.isdisjoint(valid_ids), "Train/validation case overlap detected"
assert train_ids.isdisjoint(test_ids), "Train/test case overlap detected"
assert valid_ids.isdisjoint(test_ids), "Validation/test case overlap detected"

print("Dataset validation passed.")
print(f"Train: {len(train_records)}  Validation: {len(valid_records)}  Test: {len(test_records)}")


Dataset validation passed.
Train: 2306  Validation: 466  Test: 499


In [5]:
# CELL 4 - Load Model 2 V1 (Qwen2.5-3B-Instruct + checkpoint-145), inference only
if not Path(CHECKPOINT_145_DIR).exists():
    raise FileNotFoundError(
        f"checkpoint-145 not found at {CHECKPOINT_145_DIR}. Upload the adapter directory "
        "(adapter_model.safetensors, adapter_config.json, tokenizer files) before continuing."
    )

# --- Compatibility fix: older tokenizer_config.json stored "extra_special_tokens" as a
# list, but this transformers version expects a dict. Patch a COPY, never checkpoint-145
# itself, so the original adapter directory stays untouched. ---
CHECKPOINT_145_TOKENIZER_DIR = "/content/checkpoint-145-tokenizer-fixed"
if not Path(CHECKPOINT_145_TOKENIZER_DIR).exists():
    shutil.copytree(CHECKPOINT_145_DIR, CHECKPOINT_145_TOKENIZER_DIR)

tok_cfg_path = Path(CHECKPOINT_145_TOKENIZER_DIR) / "tokenizer_config.json"
if tok_cfg_path.exists():
    with open(tok_cfg_path, "r", encoding="utf-8") as f:
        tok_cfg = json.load(f)
    if isinstance(tok_cfg.get("extra_special_tokens"), list):
        print(f"Patching extra_special_tokens (list -> dict) in copied tokenizer config "
              f"at {tok_cfg_path} - checkpoint-145 itself is unmodified.")
        tok_cfg["extra_special_tokens"] = {}
        with open(tok_cfg_path, "w", encoding="utf-8") as f:
            json.dump(tok_cfg, f, indent=2, ensure_ascii=False)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

v1_tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_145_TOKENIZER_DIR, use_fast=True)
if v1_tokenizer.pad_token is None:
    v1_tokenizer.pad_token = v1_tokenizer.eos_token
v1_tokenizer.padding_side = "left"

v1_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto", torch_dtype=compute_dtype,
)
v1_base_model.config.use_cache = True  # inference only

# The LoRA adapter weights themselves (adapter_model.safetensors, adapter_config.json)
# are loaded from the real, unmodified checkpoint-145 directory.
v1_model = PeftModel.from_pretrained(v1_base_model, CHECKPOINT_145_DIR)
v1_model.eval()

print("Loaded Qwen2.5-3B-Instruct + checkpoint-145 (Model 2 V1) for inference only.")
print("Base weights are untouched; checkpoint-145 has not been written to.")


Patching extra_special_tokens (list -> dict) in copied tokenizer config at /content/checkpoint-145-tokenizer-fixed/tokenizer_config.json - checkpoint-145 itself is unmodified.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'lora_ga_config', 'monteclora_config', 'peft_version', 'use_bdlora', 'velora_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Loaded Qwen2.5-3B-Instruct + checkpoint-145 (Model 2 V1) for inference only.
Base weights are untouched; checkpoint-145 has not been written to.


In [6]:
# CELL 5 - Exact production prompt builder (matches the confirmed training-time serialization)
def canonical_user_json(case_input: Dict[str, Any]) -> str:
    return json.dumps(
        case_input, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False,
    )

def build_prompt(case_input: Dict[str, Any], tok) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_EXPECTED},
        {"role": "user", "content": canonical_user_json(case_input)},
    ]
    # No assistant target included - inference-time prompt construction only.
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def extract_json_object(text: str) -> str:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text.strip(), flags=re.IGNORECASE).strip()
        text = re.sub(r"```$", "", text.strip()).strip()
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError(f"Could not find JSON object in model output: {text[:400]}")
    return text[start:end + 1]

def validate_prediction_schema(pred: Dict[str, Any], eligible_actions: List[str]) -> None:
    missing = REQUIRED_TARGET_KEYS - set(pred.keys())
    if missing:
        raise ValueError(f"Missing prediction keys: {sorted(missing)}")
    if pred["primary_reason"] not in APPROVED_REASONS_SET:
        raise ValueError(f"Invalid primary_reason: {pred['primary_reason']}")
    if pred["urgency"] not in APPROVED_URGENCY_SET:
        raise ValueError(f"Invalid urgency: {pred['urgency']}")
    if pred["recommended_action"] not in APPROVED_ACTIONS_SET:
        raise ValueError(f"Invalid recommended_action: {pred['recommended_action']}")
    if pred["recommended_action"] not in eligible_actions:
        raise ValueError(f"recommended_action not in eligible_actions: {eligible_actions}")
    if not isinstance(pred["secondary_reasons"], list) or any(x not in APPROVED_REASONS_SET for x in pred["secondary_reasons"]):
        raise ValueError("secondary_reasons must be a list of approved reasons")
    if not isinstance(pred["evidence"], list):
        raise ValueError("evidence must be a list")
    if not isinstance(pred["reasoning_summary"], str) or not pred["reasoning_summary"].strip():
        raise ValueError("reasoning_summary must be a non-empty string")

@torch.no_grad()
def predict_case(case_input: Dict[str, Any], mdl, tok, max_new_tokens: int = 256, max_length: int = V1_MAX_LENGTH) -> Dict[str, Any]:
    prompt = build_prompt(case_input, tok)
    model_inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=max_length).to(mdl.device)
    t0 = time.perf_counter()
    outputs = mdl.generate(
        **model_inputs, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1,
        pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id,
    )
    latency_s = time.perf_counter() - t0
    input_len = model_inputs["input_ids"].shape[-1]
    raw_text = tok.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

    result = {"raw_text": raw_text, "parsed": None, "ok": False, "error": None, "latency_s": latency_s}
    try:
        parsed = json.loads(extract_json_object(raw_text), parse_constant=reject_nonstandard_constants)
        validate_prediction_schema(parsed, case_input["eligible_actions"])
        result["parsed"], result["ok"] = parsed, True
    except Exception as e:
        result["error"] = str(e)
    return result

_sample_prompt = build_prompt(test_records[0]["user_content"], v1_tokenizer)
print(_sample_prompt[:800])
print("...")
print(f"\n(Full rendered prompt length: {len(_sample_prompt)} chars)")


<|im_start|>system
You are a banking retention analyst. Given a customer retention case, identify the most supported reason for risk, cite evidence, assign urgency, and select exactly one eligible retention action.<|im_end|>
<|im_start|>user
{"behavior":{"app_login_change_30d":-48.6909,"balance_change_30d":-75.0207,"card_spend_change_30d":-21.3437,"days_since_last_transaction":23,"emi_bounce_30d":0,"external_transfer_change_30d":83.1242,"fd_maturing_in_30d":0,"products_dropped_90d":0,"salary_missing_days":6.0,"transaction_change_30d":-51.9482,"upi_share_of_spend":0.7968},"customer_context":{"age":50,"customer_segment":"salary","customer_yearly_value":56997.527,"has_credit_card":0,"has_loan":0,"income_regularity":"regular","products_count":1,"tenure_months":125},"eligible_actions":["MONITOR
...

(Full rendered prompt length: 1313 chars)


In [7]:
# CELL 6 - Load existing V1 test evaluation (499-case benchmark). Not re-run.
existing_pred_path = os.path.join(EVAL_OUTPUT_DIR, "model2_checkpoint145_test_predictions.csv")

if Path(existing_pred_path).exists():
    print(f"Found existing V1 predictions at {existing_pred_path}; using them directly.")
    v1_pred_df = pd.read_csv(existing_pred_path)
    v1_task_metrics = {
        "num_test_cases": len(v1_pred_df),
        "json_valid_rate": v1_pred_df["schema_valid"].mean(),
        "primary_reason_exact_match": v1_pred_df["reason_match"].mean(),
        "recommended_action_exact_match": v1_pred_df["action_match"].mean(),
        "urgency_exact_match": v1_pred_df["urgency_match"].mean(),
        "invalid_unapproved_action_rate": v1_pred_df["unapproved_action"].mean(),
        "mean_latency_s": v1_pred_df["latency_s"].mean(),
        "median_latency_s": v1_pred_df["latency_s"].median(),
        "p95_latency_s": v1_pred_df["latency_s"].quantile(0.95),
    }
    V1_SOURCE = "existing_predictions_csv"
else:
    print("No existing V1 predictions CSV found in this environment.")
    print("Loading the confirmed V1 499-case results from the previously executed evaluation run.\n")

    v1_task_metrics = {
        "num_test_cases": 499,
        "json_valid_rate": 0.9920,
        "primary_reason_exact_match": 0.8557,
        "recommended_action_exact_match": 0.8918,
        "urgency_exact_match": 0.8697,
        "invalid_unapproved_action_rate": 0.0000,
        "mean_latency_s": 17.7088,
        "median_latency_s": 18.1709,
        "p95_latency_s": 20.2209,
    }

    v1_reason_recall = pd.DataFrame([
        {"label": "SERVICE_DISSATISFACTION",   "support": 146, "correct": 141, "recall": 0.965753},
        {"label": "COMPETITOR_MIGRATION",      "support": 52,  "correct": 46,  "recall": 0.884615},
        {"label": "FEE_DISSATISFACTION",       "support": 86,  "correct": 86,  "recall": 1.000000},
        {"label": "LOW_ENGAGEMENT",            "support": 23,  "correct": 14,  "recall": 0.608696},
        {"label": "PRODUCT_MISMATCH",          "support": 30,  "correct": 13,  "recall": 0.433333},
        {"label": "DIGITAL_FRICTION",          "support": 65,  "correct": 57,  "recall": 0.876923},
        {"label": "FINANCIAL_STRESS",          "support": 31,  "correct": 31,  "recall": 1.000000},
        {"label": "LIFE_STAGE_CHANGE",         "support": 24,  "correct": 24,  "recall": 1.000000},
        {"label": "TEMPORARY_SEASONAL_CHANGE", "support": 8,   "correct": 0,   "recall": 0.000000},
        {"label": "UNKNOWN",                   "support": 34,  "correct": 15,  "recall": 0.441176},
    ])

    v1_action_recall = pd.DataFrame([
        {"label": "MONITOR",               "support": 42,  "correct": 25,  "recall": 0.595238},
        {"label": "SERVICE_RECOVERY",       "support": 96,  "correct": 90,  "recall": 0.937500},
        {"label": "COMPLAINT_ESCALATION",   "support": 114, "correct": 114, "recall": 1.000000},
        {"label": "FEE_WAIVER_REVIEW",      "support": 86,  "correct": 86,  "recall": 1.000000},
        {"label": "RM_CALLBACK",            "support": 49,  "correct": 47,  "recall": 0.959184},
        {"label": "PRODUCT_REVIEW",         "support": 49,  "correct": 39,  "recall": 0.795918},
        {"label": "CARD_REVIEW",            "support": 4,   "correct": 0,   "recall": 0.000000},
        {"label": "LOAN_REVIEW",            "support": 36,  "correct": 30,  "recall": 0.833333},
        {"label": "RE_ENGAGEMENT",          "support": 23,  "correct": 14,  "recall": 0.608696},
        {"label": "FINANCIAL_GUIDANCE",     "support": 0,   "correct": 0,   "recall": float("nan")},
    ])

    v1_urgency_confusion = pd.DataFrame(
        [[8, 20, 1], [1, 190, 18], [0, 21, 236]],
        index=["LOW", "MEDIUM", "HIGH"], columns=["LOW", "MEDIUM", "HIGH"],
    )

    v1_reason_confusion_top_pairs = {
        ("COMPETITOR_MIGRATION", "LIFE_STAGE_CHANGE"): 5,
        ("LOW_ENGAGEMENT", "COMPETITOR_MIGRATION"): 5,
        ("PRODUCT_MISMATCH", "SERVICE_DISSATISFACTION"): 7,
        ("PRODUCT_MISMATCH", "COMPETITOR_MIGRATION"): 1,
        ("PRODUCT_MISMATCH", "DIGITAL_FRICTION"): 2,
        ("PRODUCT_MISMATCH", "LIFE_STAGE_CHANGE"): 2,
        ("UNKNOWN", "SERVICE_DISSATISFACTION"): 7,
        ("UNKNOWN", "LOW_ENGAGEMENT"): 3,
        ("UNKNOWN", "DIGITAL_FRICTION"): 4,
        ("UNKNOWN", "PRODUCT_MISMATCH"): 1,
        ("UNKNOWN", "COMPETITOR_MIGRATION"): 1,
        ("TEMPORARY_SEASONAL_CHANGE", "LOW_ENGAGEMENT"): 1,
    }

    v1_action_confusion_top_pairs = {
        ("MONITOR", "SERVICE_RECOVERY"): 11,
        ("MONITOR", "RM_CALLBACK"): 1,
        ("MONITOR", "PRODUCT_REVIEW"): 1,
        ("PRODUCT_REVIEW", "MONITOR"): 2,
        ("PRODUCT_REVIEW", "SERVICE_RECOVERY"): 5,
        ("PRODUCT_REVIEW", "RE_ENGAGEMENT"): 2,
        ("CARD_REVIEW", "MONITOR"): 1,
        ("CARD_REVIEW", "SERVICE_RECOVERY"): 3,
        ("LOAN_REVIEW", "MONITOR"): 1,
        ("LOAN_REVIEW", "SERVICE_RECOVERY"): 2,
        ("LOAN_REVIEW", "FINANCIAL_GUIDANCE"): 1,
        ("RE_ENGAGEMENT", "MONITOR"): 4,
        ("RE_ENGAGEMENT", "RM_CALLBACK"): 5,
        ("SERVICE_RECOVERY", "LOAN_REVIEW"): 5,
        ("SERVICE_RECOVERY", "RM_CALLBACK"): 1,
    }

    v1_fallback_stats = {
        "fallback_count": 19,
        "fallback_rate_among_valid_outputs": 0.0384,
        "fallback_rate_among_true_competitor_migration": 0.0192,
        "fallback_rate_among_true_unknown": 0.4545,
    }

    v1_competitor_migration_deep_dive = {
        "num_competitor_migration_cases": 52, "correct_primary_reason": 46,
        "correct_recommended_action": 48, "correct_urgency": 38,
        "fallback_triple_count": 1, "confused_with_UNKNOWN": 1, "confused_with_LIFE_STAGE_CHANGE": 5,
    }

    v1_segment_breakdown = pd.DataFrame([
        {"customer_segment": "salary",   "num_cases": 166, "reason_accuracy": 0.909639, "action_accuracy": 0.927711, "urgency_accuracy": 0.921687},
        {"customer_segment": "pension",  "num_cases": 113, "reason_accuracy": 0.769912, "action_accuracy": 0.849558, "urgency_accuracy": 0.831858},
        {"customer_segment": "farmer",   "num_cases": 101, "reason_accuracy": 0.891089, "action_accuracy": 0.900990, "urgency_accuracy": 0.831683},
        {"customer_segment": "vendor",   "num_cases": 69,  "reason_accuracy": 0.869565, "action_accuracy": 0.884058, "urgency_accuracy": 0.884058},
        {"customer_segment": "business", "num_cases": 50,  "reason_accuracy": 0.780000, "action_accuracy": 0.860000, "urgency_accuracy": 0.840000},
    ])

    v1_risk_breakdown = pd.DataFrame([
        {"risk_level": "Low",    "num_cases": 248, "reason_accuracy": 0.766129, "action_accuracy": 0.826613, "urgency_accuracy": 0.822581},
        {"risk_level": "Medium", "num_cases": 86,  "reason_accuracy": 0.895349, "action_accuracy": 0.918605, "urgency_accuracy": 0.883721},
        {"risk_level": "High",   "num_cases": 165, "reason_accuracy": 0.969697, "action_accuracy": 0.975758, "urgency_accuracy": 0.933333},
    ])

    v1_evidence_grounding = {
        "num_valid_outputs_checked": 495, "case_fully_grounded_rate": 0.4990,
        "total_evidence_items": 1991, "unsupported_evidence_items": 346,
    }

    V1_SOURCE = "parsed_from_provided_evaluation_notebook_outputs"

print("V1 (checkpoint-145) 499-case task metrics")
for k, v in v1_task_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
print(f"\nSource: {V1_SOURCE}")


No existing V1 predictions CSV found in this environment.
Loading the confirmed V1 499-case results from the previously executed evaluation run.

V1 (checkpoint-145) 499-case task metrics
  num_test_cases: 499
  json_valid_rate: 0.9920
  primary_reason_exact_match: 0.8557
  recommended_action_exact_match: 0.8918
  urgency_exact_match: 0.8697
  invalid_unapproved_action_rate: 0.0000
  mean_latency_s: 17.7088
  median_latency_s: 18.1709
  p95_latency_s: 20.2209

Source: parsed_from_provided_evaluation_notebook_outputs


In [8]:
# CELL 7 - V1 error analysis (uses the 499-case test results loaded/parsed in Cell 6 - no inference run here)
print("=" * 70); print("RISK-LEVEL BREAKDOWN"); print("=" * 70)
display(v1_risk_breakdown)

print("\n" + "=" * 70); print("CUSTOMER-SEGMENT BREAKDOWN"); print("=" * 70)
display(v1_segment_breakdown)

print("\n" + "=" * 70); print("URGENCY CONFUSION MATRIX (rows=expected, cols=predicted)"); print("=" * 70)
display(v1_urgency_confusion)
high_to_medium = v1_urgency_confusion.loc["HIGH", "MEDIUM"]
medium_to_high = v1_urgency_confusion.loc["MEDIUM", "HIGH"]
n_high = v1_urgency_confusion.loc["HIGH"].sum()
n_medium = v1_urgency_confusion.loc["MEDIUM"].sum()
print(f"\nHIGH -> MEDIUM: {high_to_medium}/{n_high} ({high_to_medium/n_high:.4f})")
print(f"MEDIUM -> HIGH: {medium_to_high}/{n_medium} ({medium_to_high/n_medium:.4f})")

print("\n" + "=" * 70); print("REASON CONFUSION - key pairs"); print("=" * 70)
for (exp, pred), count in v1_reason_confusion_top_pairs.items():
    if count > 0:
        print(f"  {exp:28s} -> {pred:28s} : {count}")

print("\nCOMPETITOR_MIGRATION deep dive:")
for k, v in v1_competitor_migration_deep_dive.items():
    print(f"  {k}: {v}")

print("\n" + "=" * 70); print("ACTION CONFUSION - key pairs"); print("=" * 70)
for (exp, pred), count in v1_action_confusion_top_pairs.items():
    if count > 0:
        print(f"  {exp:22s} -> {pred:22s} : {count}")

print("\n" + "=" * 70); print("FALLBACK (UNKNOWN, MEDIUM, MONITOR) STATS"); print("=" * 70)
for k, v in v1_fallback_stats.items():
    print(f"  {k}: {v}")

print("\n" + "=" * 70); print("LOWEST-RECALL CLASSES (primary_reason)"); print("=" * 70)
display(v1_reason_recall.sort_values("recall"))

print("\n" + "=" * 70); print("LOWEST-RECALL CLASSES (recommended_action)"); print("=" * 70)
display(v1_action_recall.sort_values("recall", na_position="first"))


RISK-LEVEL BREAKDOWN


,risk_level,num_cases,reason_accuracy,action_accuracy,urgency_accuracy
0,Low,248,0.766129,0.826613,0.822581
1,Medium,86,0.895349,0.918605,0.883721
2,High,165,0.969697,0.975758,0.933333



CUSTOMER-SEGMENT BREAKDOWN


,customer_segment,num_cases,reason_accuracy,action_accuracy,urgency_accuracy
0,salary,166,0.909639,0.927711,0.921687
1,pension,113,0.769912,0.849558,0.831858
2,farmer,101,0.891089,0.900990,0.831683
3,vendor,69,0.869565,0.884058,0.884058
4,business,50,0.780000,0.860000,0.840000



URGENCY CONFUSION MATRIX (rows=expected, cols=predicted)


,LOW,MEDIUM,HIGH
LOW,8,20,1
MEDIUM,1,190,18
HIGH,0,21,236



HIGH -> MEDIUM: 21/257 (0.0817)
MEDIUM -> HIGH: 18/209 (0.0861)

REASON CONFUSION - key pairs
  COMPETITOR_MIGRATION         -> LIFE_STAGE_CHANGE            : 5
  LOW_ENGAGEMENT               -> COMPETITOR_MIGRATION         : 5
  PRODUCT_MISMATCH             -> SERVICE_DISSATISFACTION      : 7
  PRODUCT_MISMATCH             -> COMPETITOR_MIGRATION         : 1
  PRODUCT_MISMATCH             -> DIGITAL_FRICTION             : 2
  PRODUCT_MISMATCH             -> LIFE_STAGE_CHANGE            : 2
  UNKNOWN                      -> SERVICE_DISSATISFACTION      : 7
  UNKNOWN                      -> LOW_ENGAGEMENT               : 3
  UNKNOWN                      -> DIGITAL_FRICTION             : 4
  UNKNOWN                      -> PRODUCT_MISMATCH             : 1
  UNKNOWN                      -> COMPETITOR_MIGRATION         : 1
  TEMPORARY_SEASONAL_CHANGE    -> LOW_ENGAGEMENT               : 1

COMPETITOR_MIGRATION deep dive:
  num_competitor_migration_cases: 52
  correct_primary_reason: 46
  

,label,support,correct,recall
8,TEMPORARY_SEASONAL_CHANGE,8,0,0.000000
4,PRODUCT_MISMATCH,30,13,0.433333
9,UNKNOWN,34,15,0.441176
3,LOW_ENGAGEMENT,23,14,0.608696
5,DIGITAL_FRICTION,65,57,0.876923
1,COMPETITOR_MIGRATION,52,46,0.884615
0,SERVICE_DISSATISFACTION,146,141,0.965753
2,FEE_DISSATISFACTION,86,86,1.000000
7,LIFE_STAGE_CHANGE,24,24,1.000000
6,FINANCIAL_STRESS,31,31,1.000000



LOWEST-RECALL CLASSES (recommended_action)


,label,support,correct,recall
9,FINANCIAL_GUIDANCE,0,0,NaN
6,CARD_REVIEW,4,0,0.000000
0,MONITOR,42,25,0.595238
8,RE_ENGAGEMENT,23,14,0.608696
5,PRODUCT_REVIEW,49,39,0.795918
7,LOAN_REVIEW,36,30,0.833333
1,SERVICE_RECOVERY,96,90,0.937500
4,RM_CALLBACK,49,47,0.959184
3,FEE_WAIVER_REVIEW,86,86,1.000000
2,COMPLAINT_ESCALATION,114,114,1.000000


In [9]:
# CELL 8 - Decide whether refinement is justified
decision_evidence = []

pm_recall = v1_reason_recall.loc[v1_reason_recall.label == "PRODUCT_MISMATCH", "recall"].iloc[0]
if pm_recall < 0.60:
    decision_evidence.append(f"PRODUCT_MISMATCH recall is {pm_recall:.2f} on 30 support cases, with a "
                              f"consistent leak pattern into 3+ other reason classes.")

unk_recall = v1_reason_recall.loc[v1_reason_recall.label == "UNKNOWN", "recall"].iloc[0]
if unk_recall < 0.60:
    decision_evidence.append(f"UNKNOWN recall is {unk_recall:.2f} on 34 support cases.")

cm_confusion = v1_reason_confusion_top_pairs.get(("COMPETITOR_MIGRATION", "LIFE_STAGE_CHANGE"), 0)
cm_support = v1_reason_recall.loc[v1_reason_recall.label == "COMPETITOR_MIGRATION", "support"].iloc[0]
if cm_confusion / cm_support > 0.05:
    decision_evidence.append(f"COMPETITOR_MIGRATION -> LIFE_STAGE_CHANGE confusion in "
                              f"{cm_confusion}/{cm_support} ({cm_confusion/cm_support:.2%}) of true "
                              f"COMPETITOR_MIGRATION cases - a known, structurally confusable pair.")

if high_to_medium / n_high > 0.05:
    decision_evidence.append(f"HIGH urgency is downgraded to MEDIUM in {high_to_medium}/{n_high} "
                              f"({high_to_medium/n_high:.2%}) of true-HIGH cases.")

mon_recall = v1_action_recall.loc[v1_action_recall.label == "MONITOR", "recall"].iloc[0]
if mon_recall < 0.70:
    decision_evidence.append(f"MONITOR action recall is {mon_recall:.2f} on 42 support cases, "
                              f"leaking mainly into SERVICE_RECOVERY (over-escalation).")

print("Evidence for refinement:")
for e in decision_evidence:
    print(f"  - {e}")

if len(decision_evidence) >= 3:
    REFINEMENT_JUSTIFIED = True
    print("\nTARGETED LORA REFINEMENT JUSTIFIED")
else:
    REFINEMENT_JUSTIFIED = False
    print("\nNO TARGETED RETRAINING JUSTIFIED")


Evidence for refinement:
  - PRODUCT_MISMATCH recall is 0.43 on 30 support cases, with a consistent leak pattern into 3+ other reason classes.
  - UNKNOWN recall is 0.44 on 34 support cases.
  - COMPETITOR_MIGRATION -> LIFE_STAGE_CHANGE confusion in 5/52 (9.62%) of true COMPETITOR_MIGRATION cases - a known, structurally confusable pair.
  - HIGH urgency is downgraded to MEDIUM in 21/257 (8.17%) of true-HIGH cases.
  - MONITOR action recall is 0.60 on 42 support cases, leaking mainly into SERVICE_RECOVERY (over-escalation).

TARGETED LORA REFINEMENT JUSTIFIED


In [10]:
# CELL 9 - Derive REAL top error patterns from the ALREADY-COMPUTED 466 validation results
# (loaded from the attached file - the 466-case set is NEVER re-run for inference here),
# then build a small, targeted, contrastive correction set from TRAIN only (never valid/test).

if REFINEMENT_JUSTIFIED:
    val_results_raw = load_jsonl_strict(VALID_RESULTS_FILE)
    val_results_df = pd.DataFrame(val_results_raw)
    if len(val_results_df) != EXPECTED_VALID_COUNT:
        print(f"WARNING: expected {EXPECTED_VALID_COUNT} validation results, found {len(val_results_df)}.")

    val_results_df["reason_match"] = val_results_df["predicted_reason"] == val_results_df["expected_reason"]
    val_results_df["action_match"] = val_results_df["predicted_action"] == val_results_df["expected_action"]
    val_results_df["urgency_match"] = val_results_df["predicted_urgency"] == val_results_df["expected_urgency"]

    print("Validation task metrics (from the attached 466-result file - no inference run):")
    print(f"  schema_valid_rate: {val_results_df['schema_valid'].mean():.4f}")
    print(f"  reason accuracy:   {val_results_df['reason_match'].mean():.4f}")
    print(f"  action accuracy:   {val_results_df['action_match'].mean():.4f}")
    print(f"  urgency accuracy:  {val_results_df['urgency_match'].mean():.4f}")
    print("\nBy risk level:")
    for risk in ["Low", "Medium", "High"]:
        sub = val_results_df[val_results_df.risk_level == risk]
        if len(sub) == 0:
            continue
        print(f"  {risk:6s} n={len(sub):3d}  reason={sub.reason_match.mean():.4f}  "
              f"action={sub.action_match.mean():.4f}  urgency={sub.urgency_match.mean():.4f}")

    MIN_PATTERN_SUPPORT = 3

    def top_confusions(df, expected_col, predicted_col, match_col, min_support):
        mism = df[(~df[match_col]) & df[predicted_col].notna()]
        counts = Counter(zip(mism[expected_col], mism[predicted_col]))
        return [(pair, n) for pair, n in counts.most_common() if n >= min_support]

    reason_confusions = top_confusions(val_results_df, "expected_reason", "predicted_reason", "reason_match", MIN_PATTERN_SUPPORT)
    action_confusions = top_confusions(val_results_df, "expected_action", "predicted_action", "action_match", MIN_PATTERN_SUPPORT)
    urgency_confusions = top_confusions(val_results_df, "expected_urgency", "predicted_urgency", "urgency_match", MIN_PATTERN_SUPPORT)

    print(f"\nReal recurring REASON confusions (support >= {MIN_PATTERN_SUPPORT}):")
    for (exp, pred), n in reason_confusions:
        print(f"  {exp:28s} -> {pred:28s} : {n}")
    print(f"\nReal recurring ACTION confusions (support >= {MIN_PATTERN_SUPPORT}):")
    for (exp, pred), n in action_confusions:
        print(f"  {exp:22s} -> {pred:22s} : {n}")
    print(f"\nReal recurring URGENCY confusions (support >= {MIN_PATTERN_SUPPORT}):")
    for (exp, pred), n in urgency_confusions:
        print(f"  {exp:8s} -> {pred:8s} : {n}")

    N_PER_CLASS_PER_PATTERN = 6
    corr_rng = random.Random(SEED)
    train_by_reason, train_by_action, train_by_urgency = {}, {}, {}
    for r in train_records:
        train_by_reason.setdefault(r["assistant_content"]["primary_reason"], []).append(r)
        train_by_action.setdefault(r["assistant_content"]["recommended_action"], []).append(r)
        train_by_urgency.setdefault(r["assistant_content"]["urgency"], []).append(r)

    def sample_for_class(pool_by_label, label, n, prefer_risk="Low", already_chosen_ids=None):
        already_chosen_ids = already_chosen_ids or set()
        candidates = [r for r in pool_by_label.get(label, []) if r["case_id"] not in already_chosen_ids]
        low_first = [r for r in candidates if r["user_content"]["model1"]["risk_level"] == prefer_risk]
        rest = [r for r in candidates if r["user_content"]["model1"]["risk_level"] != prefer_risk]
        corr_rng.shuffle(low_first); corr_rng.shuffle(rest)
        return (low_first + rest)[:n]

    correction_examples, chosen_ids = [], set()
    pattern_targets_used = {"reason": [], "action": [], "urgency": []}

    for pool, confusions, kind in [(train_by_reason, reason_confusions, "reason"),
                                    (train_by_action, action_confusions, "action"),
                                    (train_by_urgency, urgency_confusions, "urgency")]:
        for (exp, pred), n in confusions:
            picked_a = sample_for_class(pool, exp, N_PER_CLASS_PER_PATTERN, already_chosen_ids=chosen_ids)
            chosen_ids.update(r["case_id"] for r in picked_a)
            picked_b = sample_for_class(pool, pred, N_PER_CLASS_PER_PATTERN, already_chosen_ids=chosen_ids)
            chosen_ids.update(r["case_id"] for r in picked_b)
            correction_examples += picked_a + picked_b
            pattern_targets_used[kind].append({"pair": f"{exp}->{pred}", "val_support": n,
                                                "train_examples_A": len(picked_a), "train_examples_B": len(picked_b)})

    seen = set()
    deduped = []
    for r in correction_examples:
        if r["case_id"] not in seen:
            seen.add(r["case_id"]); deduped.append(r)
    correction_examples = deduped

    print(f"\nTargeted correction-set size: {len(correction_examples)}")
    print("Correction-set reason distribution:",
          dict(sorted(Counter(r["assistant_content"]["primary_reason"] for r in correction_examples).items())))
else:
    correction_examples = []
    print("Refinement not justified - skipping correction-set construction.")


Validation task metrics (from the attached 466-result file - no inference run):
  schema_valid_rate: 0.9957
  reason accuracy:   0.8562
  action accuracy:   0.8884
  urgency accuracy:  0.8970

By risk level:
  Low    n=247  reason=0.7814  action=0.8259  urgency=0.8826
  Medium n= 87  reason=0.9310  action=0.9310  urgency=0.9195
  High   n=132  reason=0.9470  action=0.9773  urgency=0.9091

Real recurring REASON confusions (support >= 3):
  PRODUCT_MISMATCH             -> UNKNOWN                      : 6
  TEMPORARY_SEASONAL_CHANGE    -> LIFE_STAGE_CHANGE            : 6
  UNKNOWN                      -> SERVICE_DISSATISFACTION      : 6
  LOW_ENGAGEMENT               -> COMPETITOR_MIGRATION         : 5
  DIGITAL_FRICTION             -> UNKNOWN                      : 4
  UNKNOWN                      -> LOW_ENGAGEMENT               : 4
  COMPETITOR_MIGRATION         -> UNKNOWN                      : 3
  DIGITAL_FRICTION             -> COMPETITOR_MIGRATION         : 3
  DIGITAL_FRICTION     

In [11]:
# CELL 10 - Combine targeted correction examples with a controlled, stratified sample
# of original TRAIN data to preserve existing strong High/Medium-risk behavior.

if REFINEMENT_JUSTIFIED and len(correction_examples) > 0:
    correction_ids = {r["case_id"] for r in correction_examples}
    remaining = [r for r in train_records if r["case_id"] not in correction_ids]

    strat_rng = random.Random(SEED)
    RETENTION_MULTIPLIER = 2.0
    RETENTION_SAMPLE_SIZE = min(len(remaining), int(len(correction_examples) * RETENTION_MULTIPLIER))

    high_med = [r for r in remaining if r["user_content"]["model1"]["risk_level"] in ("High", "Medium")]
    low = [r for r in remaining if r["user_content"]["model1"]["risk_level"] == "Low"]

    n_high_med = min(len(high_med), int(RETENTION_SAMPLE_SIZE * 0.7))
    n_low = min(len(low), RETENTION_SAMPLE_SIZE - n_high_med)

    retained_examples = strat_rng.sample(high_med, n_high_med) + strat_rng.sample(low, n_low)

    refinement_records = correction_examples + retained_examples
    strat_rng.shuffle(refinement_records)

    print(f"Correction-set size:        {len(correction_examples)}")
    print(f"Retained original examples: {len(retained_examples)} ({n_high_med} High/Medium-risk, {n_low} Low-risk)")
    print(f"Final refinement-set size:  {len(refinement_records)}")

    refinement_ids = {r["case_id"] for r in refinement_records}
    assert refinement_ids.isdisjoint(valid_ids), "Refinement set leaked validation case_ids"
    assert refinement_ids.isdisjoint(test_ids), "Refinement set leaked test case_ids"
    print("\nConfirmed: refinement set contains zero validation or test case_ids.")
else:
    refinement_records = []
    print("Refinement not justified, or no correction examples found - skipping refinement-dataset construction.")


Correction-set size:        276
Retained original examples: 552 (386 High/Medium-risk, 166 Low-risk)
Final refinement-set size:  828

Confirmed: refinement set contains zero validation or test case_ids.


In [12]:
# CELL 11 - Build refine_ds (prompt/completion message format) from refinement_records.
# This is a real, persisted step (not built ad hoc in memory) so every fresh runtime
# reproduces the identical 828-example training set from the same refinement_records.
def record_to_prompt_completion(rec):
    user_json = canonical_user_json(rec["user_content"])
    assistant_json = json.dumps(
        rec["assistant_content"], ensure_ascii=False, sort_keys=True,
        separators=(",", ":"), allow_nan=False,
    )
    prompt = [
        {"role": "system", "content": SYSTEM_PROMPT_EXPECTED},
        {"role": "user", "content": user_json},
    ]
    completion = [{"role": "assistant", "content": assistant_json}]
    return {"case_id": rec["case_id"], "prompt": prompt, "completion": completion}

if REFINEMENT_JUSTIFIED and len(refinement_records) > 0:
    refine_ds = [record_to_prompt_completion(r) for r in refinement_records]
    print(f"Built refine_ds: {len(refine_ds)} examples (prompt/completion message format).")
    assert len(refine_ds) == 828, f"Expected 828-example refine_ds, found {len(refine_ds)}"
else:
    refine_ds = []
    print("No refinement records available - refine_ds is empty.")


Built refine_ds: 828 examples (prompt/completion message format).


## Cell 12 — Custom PyTorch LoRA smoke test (manual, 5 steps only)
Run this cell to confirm the custom training loop actually updates LoRA weights before
committing to the full 828-example run. **Does not** continue automatically into Cell 13.


In [13]:
# CELL 12 - Custom PyTorch LoRA smoke test ONLY (5 optimizer steps). NO SFTTrainer.
# Does NOT continue into full refinement. Does NOT save anything. checkpoint-145 untouched.
clear_cuda()

assert 'refine_ds' in globals() and len(refine_ds) == 828, (
    f"Expected 828-example refine_ds, found {len(refine_ds) if 'refine_ds' in globals() else 'missing'}. "
    "Run Cells 8-11 first (decision -> correction set -> refinement_records -> refine_ds)."
)

GRAD_ACCUM_STEPS = 8
LR = 4e-5
SMOKE_STEPS = 5

def load_trainable_v2_model():
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto", torch_dtype=compute_dtype,
    )
    base.config.use_cache = False
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
    model = PeftModel.from_pretrained(base, CHECKPOINT_145_DIR, is_trainable=True)  # reads checkpoint-145, never writes
    model.train()
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()
    return base, model

def prepare_example(row, tok, model):
    messages = row["prompt"] + row["completion"]
    rendered = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    enc = tok(rendered, return_tensors="pt", truncation=True, max_length=V1_MAX_LENGTH)
    input_ids = enc["input_ids"].to(model.device)
    attention_mask = enc["attention_mask"].to(model.device)

    prompt_rendered = tok.apply_chat_template(row["prompt"], tokenize=False, add_generation_prompt=False)
    prompt_len = tok(prompt_rendered, return_tensors="pt", truncation=True, max_length=V1_MAX_LENGTH)["input_ids"].shape[1]

    completion_mask = torch.zeros_like(input_ids, dtype=torch.bool)
    if prompt_len < input_ids.shape[1]:
        completion_mask[:, prompt_len:] = True

    labels = input_ids.clone()
    labels[~completion_mask] = -100
    if (labels != -100).sum().item() == 0:
        raise RuntimeError("No completion tokens after tokenization/truncation.")
    return input_ids, attention_mask, labels

def snapshot_lora(model):
    return {n: p.detach().clone().float() for n, p in model.named_parameters() if "lora" in n.lower()}

print("=" * 65); print("SMOKE TEST (5 optimizer steps)"); print("=" * 65)

smoke_base, smoke_model = load_trainable_v2_model()
lora_params = [p for n, p in smoke_model.named_parameters() if "lora" in n.lower() and p.requires_grad]
trainable_count = sum(p.numel() for p in lora_params)
print(f"Trainable LoRA parameters: {trainable_count:,}")
if trainable_count == 0:
    raise RuntimeError("No trainable LoRA parameters found - aborting before touching checkpoint-145.")

before_smoke = snapshot_lora(smoke_model)
smoke_opt = torch.optim.AdamW(lora_params, lr=LR, weight_decay=0.01)
smoke_opt.zero_grad(set_to_none=True)

seen = 0
for step in range(SMOKE_STEPS):
    for _ in range(GRAD_ACCUM_STEPS):
        row = refine_ds[seen % len(refine_ds)]; seen += 1
        input_ids, attention_mask, labels = prepare_example(row, v1_tokenizer, smoke_model)
        out = smoke_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels, use_cache=False)
        loss = out.loss / GRAD_ACCUM_STEPS
        if not loss.requires_grad:
            raise RuntimeError("Loss does not require grad during smoke test.")
        loss.backward()

    grads = [p.grad.detach().float() for p in lora_params if p.grad is not None]
    if not grads:
        raise RuntimeError(f"No LoRA gradients at smoke step {step+1}.")
    grad_norm = torch.norm(torch.stack([g.norm() for g in grads])).item()
    print(f"optimizer step {step+1}/{SMOKE_STEPS} | grad_norm={grad_norm:.6f}")

    smoke_opt.step()
    smoke_opt.zero_grad(set_to_none=True)

after_smoke = snapshot_lora(smoke_model)
diffs = {n: (before_smoke[n] - after_smoke[n]).abs().max().item() for n in before_smoke}
max_weight_change = max(diffs.values())
changed_tensors = sum(v > 0 for v in diffs.values())

print(f"\nMax absolute LoRA weight change: {max_weight_change:.10e}")
print(f"LoRA tensors changed: {changed_tensors}/{len(diffs)}")

if max_weight_change > 0:
    print("\nSMOKE TEST PASSED - READY FOR FULL REFINEMENT")
else:
    print("\nSMOKE TEST FAILED - DO NOT RUN FULL REFINEMENT")

del smoke_model, smoke_base, smoke_opt, before_smoke, after_smoke, lora_params
clear_cuda()
print("\ncheckpoint-145 was not modified. No model was saved. No 466/499 inference run.")


SMOKE TEST (5 optimizer steps)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'lora_ga_config', 'monteclora_config', 'peft_version', 'use_bdlora', 'velora_config'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Trainable LoRA parameters: 29,933,568


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


optimizer step 1/5 | grad_norm=2.513987
optimizer step 2/5 | grad_norm=0.176929
optimizer step 3/5 | grad_norm=0.304388
optimizer step 4/5 | grad_norm=0.280528
optimizer step 5/5 | grad_norm=0.303954

Max absolute LoRA weight change: 2.0074332133e-04
LoRA tensors changed: 504/504

SMOKE TEST PASSED - READY FOR FULL REFINEMENT

checkpoint-145 was not modified. No model was saved. No 466/499 inference run.


In [14]:
# ============================================================
# CHECK CURRENT GPU / T4 VRAM
# ============================================================

import torch

if not torch.cuda.is_available():
    print("❌ CUDA GPU is not available.")
else:
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    used_bytes = total_bytes - free_bytes

    print("=" * 55)
    print("GPU MEMORY STATUS")
    print("=" * 55)
    print(f"GPU:         {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:  {total_bytes / 1024**3:.2f} GB")
    print(f"Used VRAM:   {used_bytes / 1024**3:.2f} GB")
    print(f"Free VRAM:   {free_bytes / 1024**3:.2f} GB")
    print(f"Free %:      {(free_bytes / total_bytes) * 100:.1f}%")

    print("\nPyTorch allocated:",
          f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print("PyTorch reserved:",
          f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB")

GPU MEMORY STATUS
GPU:         Tesla T4
Total VRAM:  14.56 GB
Used VRAM:   6.21 GB
Free VRAM:   8.35 GB
Free %:      57.3%

PyTorch allocated: 3.92 GB
PyTorch reserved: 6.08 GB


## Cell 13 — Full 828-example refinement (manual, run ONLY after Cell 12 passes)
Starts fresh from `checkpoint-145`, uses the same verified custom PyTorch loop, saves the
new V2 adapter to `V2_OUTPUT_DIR`. `checkpoint-145` is never modified.


In [15]:
# CELL 13 - FULL 828-example refinement. Run ONLY after Cell 12 prints "SMOKE TEST PASSED".
GRAD_ACCUM_STEPS = 8
LR = 4e-5
FULL_EPOCHS = 1

print("=" * 65); print(f"FULL REFINEMENT ({len(refine_ds)} examples, {FULL_EPOCHS} epoch)"); print("=" * 65)

train_base_model, v2_model = load_trainable_v2_model()  # fresh load from checkpoint-145
lora_params = [p for n, p in v2_model.named_parameters() if "lora" in n.lower() and p.requires_grad]
print(f"Trainable LoRA parameters: {sum(p.numel() for p in lora_params):,}")

before_full = snapshot_lora(v2_model)
optimizer = torch.optim.AdamW(lora_params, lr=LR, weight_decay=0.01)
optimizer.zero_grad(set_to_none=True)

indices = list(range(len(refine_ds)))
random.Random(SEED).shuffle(indices)

running_loss, micro_count, opt_steps = 0.0, 0, 0
for idx in indices:
    row = refine_ds[idx]
    input_ids, attention_mask, labels = prepare_example(row, v1_tokenizer, v2_model)
    out = v2_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels, use_cache=False)
    loss = out.loss / GRAD_ACCUM_STEPS
    loss.backward()
    running_loss += out.loss.item()
    micro_count += 1

    if micro_count % GRAD_ACCUM_STEPS == 0:
        grads = [p.grad.detach().float() for p in lora_params if p.grad is not None]
        grad_norm = torch.norm(torch.stack([g.norm() for g in grads])).item() if grads else 0.0
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        opt_steps += 1
        if opt_steps % 10 == 0 or opt_steps == 1:
            print(f"optimizer step {opt_steps} | micro-batches {micro_count} | "
                  f"grad_norm={grad_norm:.6f} | avg_loss={running_loss/micro_count:.6f}")

if micro_count % GRAD_ACCUM_STEPS != 0:
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    opt_steps += 1

print(f"\nFinished: {opt_steps} optimizer steps, {micro_count} examples seen, "
      f"final avg_loss={running_loss/max(micro_count,1):.6f}")

after_full = snapshot_lora(v2_model)
diffs = {n: (before_full[n] - after_full[n]).abs().max().item() for n in before_full}
full_max_diff = max(diffs.values())
changed_tensors = sum(v > 0 for v in diffs.values())

print(f"Max absolute LoRA weight change: {full_max_diff:.10e}")
print(f"LoRA tensors changed: {changed_tensors}/{len(diffs)}")

if full_max_diff <= 0.0:
    raise RuntimeError("Full refinement produced zero LoRA weight change - not saving V2 adapter.")

os.makedirs(V2_OUTPUT_DIR, exist_ok=True)
v2_model.save_pretrained(V2_OUTPUT_DIR)
v1_tokenizer.save_pretrained(V2_OUTPUT_DIR)
print(f"\nSaved V2 adapter + tokenizer to: {V2_OUTPUT_DIR}")

V2_TRAINED = True
del before_full, after_full, train_base_model, v2_model, optimizer
clear_cuda()
print("checkpoint-145 was not modified. 466-validation and 499-test sets were not touched.")


FULL REFINEMENT (828 examples, 1 epoch)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Trainable LoRA parameters: 29,933,568
optimizer step 1 | micro-batches 8 | grad_norm=2.527188 | avg_loss=0.118231
optimizer step 10 | micro-batches 80 | grad_norm=0.397641 | avg_loss=0.026014
optimizer step 20 | micro-batches 160 | grad_norm=0.259525 | avg_loss=0.023588
optimizer step 30 | micro-batches 240 | grad_norm=0.375849 | avg_loss=0.022387
optimizer step 40 | micro-batches 320 | grad_norm=0.260237 | avg_loss=0.021379
optimizer step 50 | micro-batches 400 | grad_norm=0.297332 | avg_loss=0.020889
optimizer step 60 | micro-batches 480 | grad_norm=0.202576 | avg_loss=0.020125
optimizer step 70 | micro-batches 560 | grad_norm=0.432282 | avg_loss=0.019834
optimizer step 80 | micro-batches 640 | grad_norm=0.190117 | avg_loss=0.019752
optimizer step 90 | micro-batches 720 | grad_norm=0.363654 | avg_loss=0.019065
optimizer step 100 | micro-batches 800 | grad_norm=0.268252 | avg_loss=0.018315

Finished: 104 optimizer steps, 828 examples seen, final avg_loss=0.018283
Max absolute LoRA wei

In [16]:
# ============================================================
# SAVE + DOWNLOAD V2 ADAPTER LOCALLY
# ============================================================

import os
import shutil
from google.colab import files

V2_DIR = "/content/model2_v2_finetuned"
ZIP_BASE = "/content/model2_v2_finetuned"

if not os.path.exists(V2_DIR):
    raise FileNotFoundError(f"V2 directory not found: {V2_DIR}")

# Create ZIP
zip_path = shutil.make_archive(
    ZIP_BASE,
    "zip",
    root_dir=V2_DIR
)

print(f"✅ V2 adapter found: {V2_DIR}")
print(f"✅ ZIP created: {zip_path}")
print(f"✅ ZIP size: {os.path.getsize(zip_path) / (1024**2):.2f} MB")

# Download to your computer
files.download(zip_path)

print("\n✅ Download started.")
print("Keep this ZIP safely — this is your genuinely updated V2 adapter.")

✅ V2 adapter found: /content/model2_v2_finetuned
✅ ZIP created: /content/model2_v2_finetuned.zip
✅ ZIP size: 109.49 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Download started.
Keep this ZIP safely — this is your genuinely updated V2 adapter.


In [17]:
# CELL 14 - Verify saved V2 adapter loads, and diff against checkpoint-145 (no eval inference)
import safetensors.torch as st

clear_cuda()

if 'V2_TRAINED' not in globals() or not V2_TRAINED:
    print("V2 was not trained in this session - skipping load/verify.")
else:
    v2_tokenizer = AutoTokenizer.from_pretrained(V2_OUTPUT_DIR, use_fast=True)
    if v2_tokenizer.pad_token is None:
        v2_tokenizer.pad_token = v2_tokenizer.eos_token
    v2_tokenizer.padding_side = "left"

    v2_base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto", torch_dtype=compute_dtype,
    )
    v2_base_model.config.use_cache = True
    v2_infer_model = PeftModel.from_pretrained(v2_base_model, V2_OUTPUT_DIR)
    v2_infer_model.eval()
    print("V2 adapter loaded successfully for inference.")

    v1_weights = st.load_file(os.path.join(CHECKPOINT_145_DIR, "adapter_model.safetensors"))
    v2_weights = st.load_file(os.path.join(V2_OUTPUT_DIR, "adapter_model.safetensors"))
    common_keys = set(v1_weights) & set(v2_weights)
    assert common_keys, "No overlapping LoRA tensor names between checkpoint-145 and V2 - architecture mismatch."

    max_diff = max((v1_weights[k].float() - v2_weights[k].float()).abs().max().item() for k in common_keys)
    print(f"Common LoRA tensors compared: {len(common_keys)}")
    print(f"Max absolute weight difference (checkpoint-145 vs V2): {max_diff:.10e}")
    print("checkpoint-145 untouched. No 466/499 inference run in this cell.")


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

V2 adapter loaded successfully for inference.
Common LoRA tensors compared: 504
Max absolute weight difference (checkpoint-145 vs V2): 2.3940103129e-03
checkpoint-145 untouched. No 466/499 inference run in this cell.


In [19]:
# CELL 15 — 100-case stratified V2 SCREENING against reused V1 validation results.
# NOT the official 466-case validation result. No retraining. No 499-test-set touch.
# Remaining 366 validation cases are intentionally NOT run.

N_SCREEN = 100
screen_rng = random.Random(SEED)

assert 'val_results_df' in globals(), "val_results_df missing — run Cell 9 (loads model2_validation_466_results.jsonl) first."
assert 'v2_infer_model' in globals() and 'v2_tokenizer' in globals(), "V2 model/tokenizer not loaded — run Cell 14 first."

valid_by_id = {r["case_id"]: r for r in valid_records}
val_results_df = val_results_df.copy()
val_results_df["stratum_key"] = list(zip(
    val_results_df["risk_level"], val_results_df["expected_reason"], val_results_df["expected_action"]
))

# Stratified sample across risk_level x expected_reason x expected_action, proportional,
# with a fallback to plain risk-level stratification if a stratum is too small.
strata = val_results_df.groupby("stratum_key")
strata_sizes = strata.size().sort_values(ascending=False)

selected_ids = []
remaining_quota = N_SCREEN
strata_list = list(strata_sizes.items())

for i, (key, size) in enumerate(strata_list):
    strata_left = len(strata_list) - i
    quota = max(1, round(remaining_quota / strata_left)) if remaining_quota > 0 else 0
    quota = min(quota, size, remaining_quota)
    ids_in_stratum = val_results_df.loc[val_results_df.stratum_key == key, "case_id"].tolist()
    screen_rng.shuffle(ids_in_stratum)
    picked = ids_in_stratum[:quota]
    selected_ids.extend(picked)
    remaining_quota -= len(picked)
    if remaining_quota <= 0:
        break

# Top up if under quota due to rounding
if len(selected_ids) < N_SCREEN:
    pool = [cid for cid in val_results_df["case_id"] if cid not in set(selected_ids)]
    screen_rng.shuffle(pool)
    selected_ids.extend(pool[:N_SCREEN - len(selected_ids)])

selected_ids = selected_ids[:N_SCREEN]
screen_df = val_results_df[val_results_df.case_id.isin(selected_ids)].reset_index(drop=True)

print("=" * 70)
print(f"100-CASE VALIDATION SCREENING (NOT the official 466-case result)")
print("=" * 70)
print(f"Selected cases: {len(screen_df)}")
print("\nRisk-level distribution:")
print(screen_df.risk_level.value_counts())

# ---- Run V2 inference on ONLY these 100 cases ----
v2_rows = []
for i, cid in enumerate(selected_ids, start=1):
    rec = valid_by_id[cid]
    pred = predict_case(rec["user_content"], v2_infer_model, v2_tokenizer)
    expected = rec["assistant_content"]
    row = {"case_id": cid, "schema_valid": pred["ok"]}
    if pred["ok"]:
        parsed = pred["parsed"]
        row["predicted_reason"] = parsed["primary_reason"]
        row["predicted_action"] = parsed["recommended_action"]
        row["predicted_urgency"] = parsed["urgency"]
        row["unapproved_action"] = parsed["recommended_action"] not in rec["user_content"]["eligible_actions"]
    else:
        row.update({"predicted_reason": None, "predicted_action": None, "predicted_urgency": None, "unapproved_action": True})
    v2_rows.append(row)
    if i % 20 == 0 or i == len(selected_ids):
        print(f"  V2 screening: {i}/{len(selected_ids)} done")

v2_screen_df = pd.DataFrame(v2_rows).rename(columns={
    "predicted_reason": "v2_reason", "predicted_action": "v2_action",
    "predicted_urgency": "v2_urgency", "schema_valid": "v2_schema_valid",
    "unapproved_action": "v2_unapproved_action",
})

merged = screen_df.merge(v2_screen_df, on="case_id", how="left")
merged = merged.rename(columns={
    "predicted_reason": "v1_reason", "predicted_action": "v1_action",
    "predicted_urgency": "v1_urgency", "schema_valid": "v1_schema_valid",
})
merged["v1_unapproved_action"] = ~merged["v1_action"].isin(APPROVED_ACTIONS)  # approximate; real flag not stored per-row upstream
if "unapproved_action" in val_results_df.columns:
    merged = merged.drop(columns=["v1_unapproved_action"]).merge(
        val_results_df[["case_id", "unapproved_action"]].rename(columns={"unapproved_action": "v1_unapproved_action"}),
        on="case_id", how="left",
    )

merged["v1_reason_match"] = merged["v1_reason"] == merged["expected_reason"]
merged["v2_reason_match"] = merged["v2_reason"] == merged["expected_reason"]
merged["v1_action_match"] = merged["v1_action"] == merged["expected_action"]
merged["v2_action_match"] = merged["v2_action"] == merged["expected_action"]
merged["v1_urgency_match"] = merged["v1_urgency"] == merged["expected_urgency"]
merged["v2_urgency_match"] = merged["v2_urgency"] == merged["expected_urgency"]

def acc(col):
    return merged[col].mean()

print("\n" + "=" * 70)
print("V1 (reused) vs V2 (fresh, 100-case screening) accuracy")
print("=" * 70)
print(f"{'metric':22s} {'V1':>8s} {'V2':>8s}")
print(f"{'WHY (reason)':22s} {acc('v1_reason_match'):8.4f} {acc('v2_reason_match'):8.4f}")
print(f"{'WHAT (action)':22s} {acc('v1_action_match'):8.4f} {acc('v2_action_match'):8.4f}")
print(f"{'URGENCY':22s} {acc('v1_urgency_match'):8.4f} {acc('v2_urgency_match'):8.4f}")
print(f"{'JSON valid':22s} {merged.v1_schema_valid.mean():8.4f} {merged.v2_schema_valid.mean():8.4f}")
print(f"{'invalid-action rate':22s} {merged.v1_unapproved_action.mean():8.4f} {merged.v2_unapproved_action.mean():8.4f}")

merged["any_diff"] = (
    (merged.v1_reason != merged.v2_reason) |
    (merged.v1_action != merged.v2_action) |
    (merged.v1_urgency != merged.v2_urgency) |
    (merged.v1_schema_valid != merged.v2_schema_valid)
)
changed = merged[merged.any_diff].copy()

def outcome(row):
    v1_correct = row.v1_reason_match and row.v1_action_match and row.v1_urgency_match and row.v1_schema_valid
    v2_correct = row.v2_reason_match and row.v2_action_match and row.v2_urgency_match and row.v2_schema_valid
    if v2_correct and not v1_correct:
        return "improvement"
    if v1_correct and not v2_correct:
        return "regression"
    return "neutral"

changed["outcome"] = changed.apply(outcome, axis=1)

print("\n" + "=" * 70)
print(f"CHANGED CASES: {len(changed)}/{len(merged)}")
print("=" * 70)
outcome_counts = changed.outcome.value_counts()
print(f"Improvements: {outcome_counts.get('improvement', 0)}")
print(f"Regressions:  {outcome_counts.get('regression', 0)}")
print(f"Neutral:      {outcome_counts.get('neutral', 0)}")

print("\n" + "=" * 70)
print("CHANGED CASES — detail (V1 vs V2 vs expected)")
print("=" * 70)
for _, r in changed.iterrows():
    print(f"\ncase_id={r.case_id}  risk={r.risk_level}  outcome={r.outcome}")
    print(f"  expected: reason={r.expected_reason:26s} action={r.expected_action:20s} urgency={r.expected_urgency}")
    print(f"  V1:       reason={str(r.v1_reason):26s} action={str(r.v1_action):20s} urgency={str(r.v1_urgency)}  schema_valid={r.v1_schema_valid}")
    print(f"  V2:       reason={str(r.v2_reason):26s} action={str(r.v2_action):20s} urgency={str(r.v2_urgency)}  schema_valid={r.v2_schema_valid}")

WATCH_REASONS = {"PRODUCT_MISMATCH", "UNKNOWN", "SERVICE_DISSATISFACTION", "DIGITAL_FRICTION", "LOW_ENGAGEMENT"}
WATCH_ACTIONS = {"MONITOR", "SERVICE_RECOVERY", "PRODUCT_REVIEW"}
WATCH_URGENCY = {"LOW", "MEDIUM", "HIGH"}

print("\n" + "=" * 70)
print("WATCHLIST CHANGES — targeted reason/action/urgency labels")
print("=" * 70)
watch_changes = changed[
    changed.v1_reason.isin(WATCH_REASONS) | changed.v2_reason.isin(WATCH_REASONS) |
    changed.v1_action.isin(WATCH_ACTIONS) | changed.v2_action.isin(WATCH_ACTIONS) |
    changed.v1_urgency.isin(WATCH_URGENCY) | changed.v2_urgency.isin(WATCH_URGENCY)
]
if len(watch_changes) == 0:
    print("  No changed cases touch the watchlist labels.")
else:
    for _, r in watch_changes.iterrows():
        print(f"  case_id={r.case_id}  outcome={r.outcome}  "
              f"reason: {r.v1_reason}->{r.v2_reason}  action: {r.v1_action}->{r.v2_action}  urgency: {r.v1_urgency}->{r.v2_urgency}")

n_improve = outcome_counts.get("improvement", 0)
n_regress = outcome_counts.get("regression", 0)
print("\n" + "=" * 70)
if n_improve > n_regress and merged.v2_schema_valid.mean() >= merged.v1_schema_valid.mean() - 0.02:
    print("V2 looks promising for final use")
else:
    print("V2 does not clearly outperform V1; keep checkpoint-145")
print("=" * 70)
print(f"\nReminder: this is a {N_SCREEN}-case SCREENING sample, not the official {EXPECTED_VALID_COUNT}-case "
      f"validation result. {EXPECTED_VALID_COUNT - N_SCREEN} validation cases were intentionally not run. "
      f"The 499-case test set was not touched.")

100-CASE VALIDATION SCREENING (NOT the official 466-case result)
Selected cases: 100

Risk-level distribution:
risk_level
Low       49
Medium    30
High      21
Name: count, dtype: int64
  V2 screening: 20/100 done
  V2 screening: 40/100 done
  V2 screening: 60/100 done
  V2 screening: 80/100 done
  V2 screening: 100/100 done

V1 (reused) vs V2 (fresh, 100-case screening) accuracy
metric                       V1       V2
WHY (reason)             0.8200   0.8400
WHAT (action)            0.8500   0.8500
URGENCY                  0.7800   0.8100
JSON valid               0.9800   1.0000
invalid-action rate      0.0200   0.0000

CHANGED CASES: 25/100
Improvements: 12
Regressions:  8
Neutral:      5

CHANGED CASES — detail (V1 vs V2 vs expected)

case_id=RC-C16860-20260601  risk=High  outcome=improvement
  expected: reason=COMPETITOR_MIGRATION       action=RM_CALLBACK          urgency=HIGH
  V1:       reason=COMPETITOR_MIGRATION       action=RM_CALLBACK          urgency=MEDIUM  schema_valid=T

In [ ]:
# CELL 16 - Final 499-case test evaluation for V1 (reused from Cell 6) and V2 (fresh, if trained)
def evaluate_on_test(mdl, tok, records):
    rows = []
    for rec in records:
        pred = predict_case(rec["user_content"], mdl, tok)
        expected = rec["assistant_content"]
        row = {
            "case_id": rec["case_id"],
            "risk_level": rec["user_content"]["model1"]["risk_level"],
            "expected_reason": expected["primary_reason"],
            "expected_action": expected["recommended_action"],
            "expected_urgency": expected["urgency"],
            "schema_valid": pred["ok"],
            "latency_s": pred["latency_s"],
        }
        if pred["ok"]:
            parsed = pred["parsed"]
            row["predicted_reason"] = parsed["primary_reason"]
            row["predicted_action"] = parsed["recommended_action"]
            row["predicted_urgency"] = parsed["urgency"]
            row["reason_match"] = parsed["primary_reason"] == expected["primary_reason"]
            row["action_match"] = parsed["recommended_action"] == expected["recommended_action"]
            row["urgency_match"] = parsed["urgency"] == expected["urgency"]
            row["unapproved_action"] = parsed["recommended_action"] not in rec["user_content"]["eligible_actions"]
        else:
            row.update({"predicted_reason": None, "predicted_action": None, "predicted_urgency": None,
                        "reason_match": False, "action_match": False, "urgency_match": False, "unapproved_action": True})
        rows.append(row)
    return pd.DataFrame(rows)

if 'V2_TRAINED' in globals() and V2_TRAINED:
    print(f"Running V2 inference on the untouched {len(test_records)}-case test set...")
    v2_test_df = evaluate_on_test(v2_infer_model, v2_tokenizer, test_records)
    v2_task_metrics = {
        "num_test_cases": len(v2_test_df),
        "json_valid_rate": v2_test_df.schema_valid.mean(),
        "primary_reason_exact_match": v2_test_df.reason_match.mean(),
        "recommended_action_exact_match": v2_test_df.action_match.mean(),
        "urgency_exact_match": v2_test_df.urgency_match.mean(),
        "invalid_unapproved_action_rate": v2_test_df.unapproved_action.mean(),
        "mean_latency_s": v2_test_df.latency_s.mean(),
        "median_latency_s": v2_test_df.latency_s.median(),
        "p95_latency_s": v2_test_df.latency_s.quantile(0.95),
    }
    print("\nV2 499-case test metrics")
    for k, v in v2_task_metrics.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")
else:
    v2_test_df, v2_task_metrics = None, None
    print("V2 was not trained - no new 499-case run needed; V1 remains the only candidate.")

print("\nV1 499-case test metrics (reused from Cell 6 - not re-run):")
for k, v in v1_task_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


In [ ]:
# CELL 17 - Final V1 vs V2 decision
DEGRADATION_TOLERANCE = 0.02

if 'V2_TRAINED' in globals() and V2_TRAINED and v2_task_metrics is not None:
    guards_ok = True
    guard_report = []

    def check_guard(name, v1_val, v2_val):
        global guards_ok
        drop = v1_val - v2_val
        ok = drop <= DEGRADATION_TOLERANCE
        guards_ok = guards_ok and ok
        guard_report.append(f"  {name}: V1={v1_val:.4f} V2={v2_val:.4f} drop={drop:.4f} -> {'OK' if ok else 'REGRESSION'}")

    check_guard("reason_accuracy", v1_task_metrics["primary_reason_exact_match"], v2_task_metrics["primary_reason_exact_match"])
    check_guard("action_accuracy", v1_task_metrics["recommended_action_exact_match"], v2_task_metrics["recommended_action_exact_match"])
    check_guard("urgency_accuracy", v1_task_metrics["urgency_exact_match"], v2_task_metrics["urgency_exact_match"])
    check_guard("json_valid_rate", v1_task_metrics["json_valid_rate"], v2_task_metrics["json_valid_rate"])

    print("Guardrail report (499-case test set):")
    for line in guard_report:
        print(line)

    if guards_ok:
        FINAL_MODEL, FINAL_MODEL_PATH = "V2", V2_OUTPUT_DIR
        DECISION_REASON = "V2 does not regress any guarded metric beyond tolerance on the 499-case test set."
    else:
        FINAL_MODEL, FINAL_MODEL_PATH = "V1", CHECKPOINT_145_DIR
        DECISION_REASON = "V2 regressed one or more guarded metrics beyond tolerance; keeping V1."
else:
    FINAL_MODEL, FINAL_MODEL_PATH = "V1", CHECKPOINT_145_DIR
    DECISION_REASON = "V2 was not trained in this session; V1 is the only available candidate."

print(f"\nFINAL MODEL: {FINAL_MODEL}")
print(f"Path: {FINAL_MODEL_PATH}")
print(f"Reason: {DECISION_REASON}")


In [ ]:
# CELL 18 - Save final outputs (scorecard)
final_scorecard = {
    "final_model": FINAL_MODEL,
    "final_model_path": FINAL_MODEL_PATH,
    "decision_reason": DECISION_REASON,
    "v1_test_metrics": v1_task_metrics,
    "v2_test_metrics": v2_task_metrics,
    "refinement_justified": REFINEMENT_JUSTIFIED,
    "v2_trained": bool(globals().get("V2_TRAINED", False)),
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}

scorecard_path = os.path.join(EVAL_OUTPUT_DIR, "model2_final_scorecard.json")
with open(scorecard_path, "w", encoding="utf-8") as f:
    json.dump(final_scorecard, f, indent=2, ensure_ascii=False, default=str)
print(f"Saved final scorecard to: {scorecard_path}")


In [ ]:
# CELL 19 - Final inference/demo function using the SELECTED model
if FINAL_MODEL == "V1":
    final_model, final_tokenizer = v1_model, v1_tokenizer
else:
    final_model, final_tokenizer = v2_infer_model, v2_tokenizer

def predict_customer_case(case_input: Dict[str, Any]) -> Dict[str, Any]:
    """Runs the selected final Model 2 on one customer case and returns a schema-enforced result."""
    if not isinstance(case_input, dict) or not REQUIRED_USER_KEYS.issubset(case_input.keys()):
        raise ValueError(f"case_input must contain keys: {sorted(REQUIRED_USER_KEYS)}")
    pred = predict_case(case_input, final_model, final_tokenizer)
    if not pred["ok"]:
        raise ValueError(f"Model output failed schema validation: {pred['error']}\nRaw: {pred['raw_text'][:500]}")
    return pred["parsed"]

print(f"Ready for inference using {FINAL_MODEL} at {FINAL_MODEL_PATH}")


In [ ]:
# CELL 20 - FINAL REPORT
print("=" * 60); print("MODEL 2 FINAL REPORT"); print("=" * 60)
print("\nBase model:\nQwen2.5-3B-Instruct")
print(f"\nSelected adapter:\n{FINAL_MODEL_PATH}")
print(f"\nTraining cases:\n{len(train_records)}")
print(f"\nValidation cases:\n{len(valid_records)} (results reused from attached file, never re-run)")
print(f"\nFinal test cases:\n{len(test_records)}")

final_metrics = v2_task_metrics if (FINAL_MODEL == "V2" and v2_task_metrics is not None) else v1_task_metrics
print(f"\nWHY accuracy:\n{final_metrics['primary_reason_exact_match']:.4f}")
print(f"\nWHAT accuracy:\n{final_metrics['recommended_action_exact_match']:.4f}")
print(f"\nUrgency accuracy:\n{final_metrics['urgency_exact_match']:.4f}")
print(f"\nValid JSON:\n{final_metrics['json_valid_rate']:.4f}")
